# Demo Notebook

This is a demo notebook to show you how to load the data from Google BigQuery into a pandas dataframe and then use the data to construct a simple visualization.

In [1]:
import sys
print(sys.executable)

/Users/nicole/data-training/.venv/bin/python


In [1]:
import plotly.express as px
import numpy as np

In [2]:
# read bigquery data into pandas dataframe
import pandas as pd

df = pd.read_gbq(
    """
    SELECT  *
    FROM `jr-data-training.cafe.cafe-sales`
    --- LIMIT 10
  """,
    project_id="jr-data-training",
    location="australia-southeast1",
)

df.head()

/var/folders/03/kg6152lx62g3z5rhmtm47_c80000gn/T/ipykernel_25814/1294536879.py:4: FutureWarning: read_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.read_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.read_gbq
  df = pd.read_gbq(


,order_id,customer_id,ip_addr,date_created,date_paid,total,status,items
0,11787,517,81741da2cc0e5207bfefbb4ec257b2393d5e13a0,2021-04-14 05:56:06,2021-04-14 05:56:37,1280,2,"{""cart_size"":2,""cart_surcharge"":0,""cart_total_..."
1,40606,519,ccf5e36bddfbce1a7de2c04874190a057b30588d,2024-03-05 10:16:40,2024-03-05 10:17:23,1280,2,"{""cart_size"":2,""cart_surcharge"":0,""cart_total_..."
2,40488,519,ccf5e36bddfbce1a7de2c04874190a057b30588d,2024-03-01 11:40:20,2024-03-01 11:40:58,1280,2,"{""cart_size"":2,""cart_surcharge"":0,""cart_total_..."
3,38252,519,1f8e636af94847a55418d2c03ccf4b1601d10512,2023-12-05 08:48:02,2023-12-05 08:49:24,1280,2,"{""cart_size"":2,""cart_surcharge"":0,""cart_total_..."
4,13470,776,d4c84c8e50cf80cc385d669000208e59e304fcca,2021-06-20 12:10:17,2021-06-20 12:11:09,1280,2,"{""cart_size"":2,""cart_surcharge"":0,""cart_total_..."


# EDA

## Initial EDA

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37372 entries, 0 to 37371
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   order_id      37372 non-null  Int64         
 1   customer_id   37372 non-null  Int64         
 2   ip_addr       37372 non-null  object        
 3   date_created  37372 non-null  datetime64[us]
 4   date_paid     36019 non-null  datetime64[us]
 5   total         37372 non-null  Int64         
 6   status        37372 non-null  Int64         
 7   items         37372 non-null  object        
 8   month         37372 non-null  period[M]     
dtypes: Int64(4), datetime64[us](2), object(2), period[M](1)
memory usage: 2.7+ MB


In [10]:
df.isnull().sum()

order_id           0
customer_id        0
ip_addr            0
date_created       0
date_paid       1353
total              0
status             0
items              0
month              0
dtype: int64

In [11]:
df.nunique()

order_id        37372
customer_id      1755
ip_addr         17630
date_created    37341
date_paid       36000
total             766
status              5
items           16278
month              59
dtype: int64

- From the above overlook status, we can see that **1353** columns are empty in **date_paid**.
- **Status** is considered as category column, as there are only 5 different values.
- The **item** column value is a JSON array, it need to be divided into more specific parts.

In [10]:
item_array = df['items'][0]

'{"cart_size":2,"cart_surcharge":0,"cart_total_price":1280,"cart_gst":116.3636363636363597606759867630898952484130859375,"cart_surcharge_display":"$0.00","cart_total_price_display":"$12.80","cart_gst_display":"$1.16","cart":[{"name":"Latte","variant_name":"Large","variant_desc":"","variant_image":"images\\/coffeecup-latte.png","options":[{"name":"size","value":"LRG","price":430},{"name":"Milk","value":"Soy","price":50},{"name":"Strength","value":"Full","price":0},{"name":"Decaf","value":"Normal","price":0},{"name":"Temp","value":"Normal","price":0},{"name":"Honey","value":"None","price":0},{"name":"Syrup","value":"None","price":0},{"name":"White Sugar","value":"0","price":0},{"name":"Raw Sugar","value":"0","price":0},{"name":"Equal Sugar","value":"0","price":0},{"name":"Extra shot","value":"0","price":0}],"price":480,"category":"Hot Drinks"},{"name":"Toastie","variant_name":"Bacon, Egg and Cheese","variant_desc":"","variant_image":"images\\/toastie-baconegg.png","options":[{"name":"siz

## Revenue Trend

### Monthly revenue trend analysis

In [34]:
df['date_created']
df['month'] = df['date_created'].dt.to_period('M')
monthly_sales = df.groupby('month')['total'].sum() / 100

fig = px.line(x=monthly_sales.index.astype(str), y=monthly_sales.values, labels={'x':'Month', 'y':'Total Sales($AUD)'}, title='Monthly Sales')
fig.show()

The trend of the monthly sales are overall fluctuated. It increased slightly from May 2019 to around Sep 2020, then the sales figures fluctuate but generally maintain a higher level compared to the initial period.

- The peak is observed in Sep 2021, with a total of 13.26K(AUD).
- Keep in mind that there are spikes around each end of the year (Oct to Dec).
- A significant drop is observed in Mar 2024.


### Average order value examination

In [56]:
AOV = df.groupby('month')['total'].mean() / 100

fig = px.line(x=AOV.index.astype(str), y=AOV.values, labels={'x':'Month', 'y':'Average Sales($AUD)'}, title='Monthly Average Sales')
fig.show()

The metric illustrates a relatively stable values with occasional peak from May 2019 to Mar 2020. Then fluctuation occurs in around Apr 2020 to Jun 2021. After reaching the highest peak, the overall trend appears to be downward. 

- The highest peak is in Sep 2021, with AOV 14.68(AUD).
- The lowest point is in May 2023, AOV 9.21(AUD).

### Identification of peak revenue hours.

In [53]:
df['hour'] = df['date_created'].dt.strftime('%H').add(':00')
hour_sales = df.groupby('hour')['total'].sum() / 100

fig = px.line(x=hour_sales.index.astype(str), y=hour_sales.values, labels={'x':'Hour', 'y':'Total Sales($AUD)'}, title='Hourly Sales')
fig.show()

In [58]:
df['hour'] = df['date_created'].dt.strftime('%H').add(':00')
hour_sales_average = df.groupby('hour')['total'].mean() / 100

fig = px.line(x=hour_sales_average.index.astype(str), y=hour_sales_average.values, labels={'x':'Hour', 'y':'Average Sales($AUD)'}, title='Hourly Average Sales')
fig.show()

Obviously, the peak revenue hour is at 8:00A.M, with a total of 92.4K revenue. On the other hand, the highest average revenue appears at midnight.